# Public Health Compacts Monitor (2026)

Live monitoring notebook for interstate public-health alliances and compacts.

### What this notebook does
1. Defines alliance/compact data (GPHA, WCHA, NEPHC, ASTHO)
2. Fetches and filters RSS/news feeds from CDC, NIH, NYT, NPR, KFF, ProPublica, and others
3. Provides empty tracker DataFrames for meetings, research, and policy entries
4. Renders a network graph of alliance–state relationships
5. Exports a fully self-contained interactive HTML dashboard

### Usage
- Run **Cell by Cell** to inspect each stage, or **Run All** for a single-pass export
- Fill in the `meetings`, `research`, and `policy` DataFrames with hand-curated entries
- The final cell writes `public_health_dashboard.html` — open it in any browser


> **v3.2 — Resilient Fetcher Update:** Feed fetching now uses `requests` with retry/backoff, SSL tolerance (`verify=False`), lxml malformed-XML recovery, and HTML fallback feed autodiscovery. Eliminates the XML parse errors and SSL certificate failures seen in v3.1.

In [1]:
# ── Additional resilient-fetching dependencies ────────────────────────────
# These enable malformed-XML recovery, HTML fallback, and SSL tolerance.
# Run once per environment; safe to re-run.
!pip install -q \
    feedparser \
    beautifulsoup4 \
    lxml \
    requests \
    urllib3


In [2]:
# ── Imports & configuration ────────────────────────────────────────────────
import json
import datetime
import pathlib
import email.utils
import textwrap

import pandas as pd
import feedparser
import networkx as nx
import matplotlib
matplotlib.use("Agg")          # headless rendering — safe in Jupyter and scripts
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Output path ────────────────────────────────────────────────────────────
OUTPUT_HTML = pathlib.Path("public_health_dashboard.html")

# ── News-fetch settings ────────────────────────────────────────────────────
MAX_ENTRIES_PER_FEED = 15      # entries fetched per feed before filtering
HEALTH_KEYWORDS = [            # at least one must appear in title or summary
    "health", "disease", "vaccine", "outbreak", "virus", "epidemic",
    "pandemic", "CDC", "NIH", "FDA", "public health", "hospital",
    "infection", "mortality", "surveillance", "preparedness",
    "emergency", "alliance", "compact", "GPHA", "WCHA", "NEPHC", "ASTHO",
]

FETCH_TIMEOUT = 10             # seconds per feed request

print("✓ imports OK —", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))


✓ imports OK — 2026-05-27 07:29


In [3]:
# ── Additional imports for resilient feed fetching ────────────────────────
import ssl
import random
import urllib3

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from lxml import etree
from urllib.parse import urljoin

urllib3.disable_warnings()


In [4]:
# ── Resilient-fetching config ─────────────────────────────────────────────
REQUEST_TIMEOUT = 30

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0 Safari/537.36"
)


def build_session():
    """Return a requests.Session with retry/backoff and browser-like headers."""
    session = requests.Session()
    retries = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": (
            "application/rss+xml,"
            "application/xml,"
            "text/xml,"
            "application/atom+xml,"
            "text/html"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    })
    return session


In [5]:
# ── Malformed-XML recovery ─────────────────────────────────────────────────
def recover_malformed_xml(raw_content):
    """Attempt lxml recovery on a malformed XML byte string; return repaired bytes."""
    try:
        parser = etree.XMLParser(recover=True)
        tree   = etree.fromstring(raw_content, parser=parser)
        return etree.tostring(tree)
    except Exception:
        return raw_content


# ── HTML feed autodiscovery ────────────────────────────────────────────────
def discover_feed_from_html(html, base_url):
    """Return a list of feed URLs found in <link> tags of an HTML page."""
    soup = BeautifulSoup(html, "html.parser")
    discovered = []
    for link in soup.find_all("link"):
        link_type = (link.get("type") or "").lower()
        href = link.get("href")
        if not href:
            continue
        if "rss" in link_type or "atom" in link_type or "xml" in link_type:
            discovered.append(urljoin(base_url, href))
    return discovered


## 1 · Alliance & Compact Data

In [6]:
# ── Alliance / compact reference data ─────────────────────────────────────
# Edit this list to add or update alliances.

ALLIANCE_RECORDS = [
    {
        "Alliance":                   "Governors Public Health Alliance (GPHA)",
        "Abbreviation":               "GPHA",
        "Founded":                    "2025",
        "Member States/Territories":  "CA, CO, CT, DE, GU, HI, IL, MA, MD, NC, NJ, NY, OR, RI, WA",
        "Primary Goals":              (
            "Cross-state public health coordination, vaccine access, "
            "emergency preparedness, disease monitoring, policy alignment"
        ),
        "Work Groups / Focus Areas":  (
            "Vaccines, outbreak detection, emergency preparedness, "
            "regulatory coordination, public messaging"
        ),
        "Cross-Collaborations":       "West Coast Health Alliance, Northeast Public Health Collaborative, GovAct, WHO/GOARN-aligned initiatives",
        "Governance Notes":           "Governor-led coalition supported by GovAct",
        "Website":                    "https://www.govsforhealth.org/",
    },
    {
        "Alliance":                   "West Coast Health Alliance (WCHA)",
        "Abbreviation":               "WCHA",
        "Founded":                    "Regional pandemic-era coordination, expanded post-2024",
        "Member States/Territories":  "CA, OR, WA",
        "Primary Goals":              "Regional disease coordination, vaccine policy harmonization, emergency response",
        "Work Groups / Focus Areas":  "Vaccines, respiratory disease surveillance, climate-health preparedness",
        "Cross-Collaborations":       "GPHA, state public health departments",
        "Governance Notes":           "West Coast governors and health agencies",
        "Website":                    "",
    },
    {
        "Alliance":                   "Northeast Public Health Collaborative (NEPHC)",
        "Abbreviation":               "NEPHC",
        "Founded":                    "Post-pandemic interstate coordination framework",
        "Member States/Territories":  "Northeast regional states (varies by initiative)",
        "Primary Goals":              "Shared surveillance, emergency planning, health communication coordination",
        "Work Groups / Focus Areas":  "Data sharing, outbreak response, laboratory coordination",
        "Cross-Collaborations":       "GPHA, regional health departments",
        "Governance Notes":           "Collaborative interstate public health network",
        "Website":                    "",
    },
    {
        "Alliance":                   "Association of State and Territorial Health Officials (ASTHO)",
        "Abbreviation":               "ASTHO",
        "Founded":                    "1942",
        "Member States/Territories":  "All U.S. states and territories",
        "Primary Goals":              "Policy coordination, public health capacity building, federal-state collaboration",
        "Work Groups / Focus Areas":  "Preparedness, chronic disease, infectious disease, workforce",
        "Cross-Collaborations":       "CDC, state health agencies, GPHA-adjacent initiatives",
        "Governance Notes":           "National nonprofit representing public health agencies",
        "Website":                    "https://www.astho.org/",
    },
]

alliance_df = pd.DataFrame(ALLIANCE_RECORDS)

# Summary display (all columns except long free-text ones)
display_cols = ["Abbreviation", "Founded", "Member States/Territories", "Governance Notes", "Website"]
print(f"✓ {len(alliance_df)} alliances loaded")
alliance_df[display_cols]


✓ 4 alliances loaded


,Abbreviation,Founded,Member States/Territories,Governance Notes,Website
0,GPHA,2025,"CA, CO, CT, DE, GU, HI, IL, MA, MD, NC, NJ, NY...",Governor-led coalition supported by GovAct,https://www.govsforhealth.org/
1,WCHA,"Regional pandemic-era coordination, expanded p...","CA, OR, WA",West Coast governors and health agencies,
2,NEPHC,Post-pandemic interstate coordination framework,Northeast regional states (varies by initiative),Collaborative interstate public health network,
3,ASTHO,1942,All U.S. states and territories,National nonprofit representing public health ...,https://www.astho.org/


## 2 · RSS / News Feed Monitoring

In [7]:
RSS_FEEDS = [
# FEDERAL PUBLIC HEALTH AGENCIES
    {"url": "https://www.cdc.gov/media/rss.xml", "label": "CDC Newsroom"},
    {"url": "https://www.nih.gov/news-events/news-releases/feed", "label": "NIH News Releases"},
    {"url": "https://tools.cdc.gov/api/v2/resources/media/403372.rss", "label": "CDC Media"},
    {"url": "https://www.cdc.gov/mmwr/index2026.xml", "label": "CDC MMWR"},
    {"url": "https://emergency.cdc.gov/rss.asp", "label": "CDC Emergency Preparedness"},
    {"url": "https://www.nih.gov/news-events/news-releases/feed", "label": "NIH News Releases"},
    {"url": "https://directorsblog.nih.gov/feed/", "label": "NIH Director Blog"},
    {"url": "https://www.fda.gov/about-fda/contact-fda/stay-informed/rss-feeds/recalls/rss.xml", "label": "FDA Recalls"},
    {"url": "https://www.fda.gov/about-fda/contact-fda/stay-informed/rss-feeds/press-announcements/rss.xml", "label": "FDA Press"},
    {"url": "https://www.cms.gov/newsroom/rss-feeds/all-news.xml", "label": "CMS News"},
    {"url": "https://www.hhs.gov/rss/index.xml", "label": "HHS News"},
    {"url": "https://aspr.hhs.gov/rss.xml", "label": "ASPR Preparedness"},
    {"url": "https://www.samhsa.gov/newsroom/rss.xml", "label": "SAMHSA"},
    {"url": "https://www.hrsa.gov/about/news/rss.xml", "label": "HRSA"},
    {"url": "https://www.ahrq.gov/news/newsroom/index.xml", "label": "AHRQ"},
    {"url": "https://www.niaid.nih.gov/news-events/rss.xml", "label": "NIAID"},
    {"url": "https://www.nimh.nih.gov/site-info/index-rss.shtml", "label": "NIMH"},
# PUBLIC HEALTH LAW + GOVERNANCE
    {"url": "https://www.networkforphl.org/rss.xml", "label": "Network for Public Health Law"},
    {"url": "https://www.healthaffairs.org/action/showFeed?type=etoc&feed=rss&jc=hlthaff", "label": "Health Affairs"},
    {"url": "https://www.commonwealthfund.org/publications/rss.xml", "label": "Commonwealth Fund"},
    {"url": "https://www.kff.org/feed/", "label": "KFF Policy"},
    {"url": "https://www.pewtrusts.org/en/rss/health-care", "label": "Pew Health Care"},
    {"url": "https://www.ncsl.org/DesktopModules/NCSLDataManager/Services/Feed.ashx?Feed=PressRoom", "label": "NCSL"},
    {"url": "https://www.ncbi.nlm.nih.gov/feed/rss.cgi?ChanKey=PubMedNews", "label": "PubMed News"},
# INTERSTATE GOVERNANCE + STATE COORDINATION
    {"url": "https://www.nga.org/news/feed/", "label": "National Governors Association"},
    {"url": "https://www.astho.org/rss.xml", "label": "ASTHO"},
    {"url": "https://www.naccho.org/blog/rss", "label": "NACCHO"},
    {"url": "https://www.csg.org/feed/", "label": "Council of State Governments"},
    {"url": "https://www.nasbo.org/rss", "label": "NASBO"},
    {"url": "https://www.ecsgroup.org/feed", "label": "Education Commission of the States"},
    {"url": "https://www.nlc.org/feed/", "label": "National League of Cities"},
# EPIDEMIOLOGY + OUTBREAK INTELLIGENCE
    {"url": "https://promedmail.org/promed-posts/rss/", "label": "ProMED"},
    {"url": "https://healthmap.org/en/rss/", "label": "HealthMap"},
    {"url": "https://www.cidrap.umn.edu/news-perspective/rss.xml", "label": "CIDRAP"},
    {"url": "https://www.statnews.com/feed/", "label": "STAT News"},
    {"url": "https://outbreaknewstoday.com/feed/", "label": "Outbreak News Today"},
    {"url": "https://www.eurekalert.org/rss/health_medicine.xml", "label": "EurekAlert Health"},
    {"url": "https://www.who.int/feeds/entity/csr/don/en/rss.xml", "label": "WHO Disease Outbreak News"},
    {"url": "https://www.ecdc.europa.eu/en/rss.xml", "label": "ECDC Europe"},
# PANDEMIC PREPAREDNESS + BIOSECURITY
    {"url": "https://www.centerforhealthsecurity.org/newsroom/rss.xml", "label": "Johns Hopkins Center for Health Security"},
    {"url": "https://www.nti.org/feed/", "label": "NTI Biosecurity"},
    {"url": "https://www.gavi.org/rss.xml", "label": "GAVI"},
    {"url": "https://cepi.net/feed/", "label": "CEPI"},
    {"url": "https://pandemicactionnetwork.org/feed/", "label": "Pandemic Action Network"},
# REPRODUCTIVE HEALTH + ABORTION POLICY
    {"url": "https://www.guttmacher.org/rss.xml", "label": "Guttmacher Institute"},
    {"url": "https://www.reproductiverights.org/feed/", "label": "Center for Reproductive Rights"},
    {"url": "https://rewirenewsgroup.com/feed/", "label": "Rewire News Group"},
    {"url": "https://abortioncarenetwork.org/feed/", "label": "Abortion Care Network"},
    {"url": "https://www.plannedparenthood.org/about-us/newsroom/feed", "label": "Planned Parenthood"},
    {"url": "https://jessica.substack.com", "label": "Abortion Every Day"},
# ENVIRONMENTAL HEALTH + CLIMATE GOVERNANCE
    {"url": "https://www.epa.gov/newsreleases/rss.xml", "label": "EPA News"},
    {"url": "https://www.climate.gov/feeds/all", "label": "NOAA Climate"},
    {"url": "https://yaleclimateconnections.org/feed/", "label": "Yale Climate Connections"},
    {"url": "https://insideclimatenews.org/feed/", "label": "Inside Climate News"},
    {"url": "https://www.edf.org/rss.xml", "label": "Environmental Defense Fund"},
    {"url": "https://www.nrdc.org/rss.xml", "label": "NRDC"},
# HEALTHCARE SYSTEM + INSURANCE POLICY
    {"url": "https://www.modernhealthcare.com/section/rss", "label": "Modern Healthcare"},
    {"url": "https://www.fiercehealthcare.com/rss/xml", "label": "Fierce Healthcare"},
    {"url": "https://www.beckershospitalreview.com/rss-lists.html", "label": "Becker Hospital Review"},
    {"url": "https://www.ajmc.com/rss.xml", "label": "AJMC"},
    {"url": "https://www.healthcaredive.com/feeds/news/", "label": "Healthcare Dive"},
# SCIENTIFIC + ACADEMIC JOURNALS
    {"url": "https://jamanetwork.com/rss/site_3/67.xml", "label": "JAMA"},
    {"url": "https://www.nejm.org/action/showFeed?type=etoc&feed=rss&jc=nejm", "label": "NEJM"},
    {"url": "https://www.thelancet.com/rssfeed/lancet_current.xml", "label": "The Lancet"},
    {"url": "https://ajph.aphapublications.org/action/showFeed?type=etoc&feed=rss&jc=ajph", "label": "AJPH"},
    {"url": "https://bmj.com/rss/news.xml", "label": "BMJ"},
    {"url": "https://www.nature.com/subjects/public-health.rss", "label": "Nature Public Health"},
    {"url": "https://www.science.org/action/showFeed?type=etoc&feed=rss&jc=science", "label": "Science"},
# HEALTH EQUITY + SOCIAL DETERMINANTS
    {"url": "https://www.rwjf.org/en/blog/_jcr_content/mainparsys/blogposts.rss", "label": "Robert Wood Johnson Foundation"},
    {"url": "https://www.commonwealthfund.org/publications/rss.xml", "label": "Commonwealth Fund"},
    {"url": "https://www.urban.org/rss.xml", "label": "Urban Institute"},
    {"url": "https://www.brookings.edu/topic/health-care/feed/", "label": "Brookings Health"},
# EMERGENCY MANAGEMENT + DISASTER RESPONSE
    {"url": "https://www.fema.gov/rss-feeds", "label": "FEMA"},
    {"url": "https://cdphe.colorado.gov/category/press-release", "label": "Ready.gov"},
    {"url": "https://www.redcross.org/about-us/news-and-events/news/rss/all.xml", "label": "American Red Cross"},
# STATE HEALTH DEPARTMENTS
    {"url": "https://www.cdph.ca.gov/RSS/Pages/default.aspx", "label": "California Department of Public Health"},
    {"url": "https://cdphe.colorado.gov/category/press-release", "label": "Colorado Public Health"},
    {"url": "https://www.nj.gov/health/news/rss.xml", "label": "New Jersey Health"},
    {"url": "https://health.ny.gov/press/releases/rss.xml", "label": "New York Health"},
    {"url": "https://doh.wa.gov/rss.xml", "label": "Washington Health"},
    {"url": "https://www.mass.gov/rss-feeds/dph-news-releases", "label": "Massachusetts DPH"},
    {"url": "https://www.oregon.gov/oha/ERD/Pages/rss.aspx", "label": "Oregon Health Authority"},
# MEDIA + INVESTIGATIVE SOURCES
    {"url": "https://www.propublica.org/feeds/propublica/main", "label": "ProPublica"},
    {"url": "https://rss.nytimes.com/services/xml/rss/nyt/Health.xml", "label": "NYT Health"},
    {"url": "https://www.theguardian.com/society/health/rss", "label": "Guardian Health"},
    {"url": "https://www.reutersagency.com/feed/?best-topics=healthcare-pharmaceuticals", "label": "Reuters Healthcare"},
    {"url": "https://kffhealthnews.org/feed", "label": "KFF Health News"},
    {"url": "https://yourlocalepidemiologist.substack.com/feed", "label": "Your Local Epidemiologist"},
    {"url": "https://yourlocalepidemiologistny.substack.com/feed", "label": "YLE – NY"},
]



print(f"✓ {len(RSS_FEEDS)} feed sources configured")
pd.DataFrame(RSS_FEEDS)


✓ 89 feed sources configured


,url,label
0,https://www.cdc.gov/media/rss.xml,CDC Newsroom
1,https://www.nih.gov/news-events/news-releases/...,NIH News Releases
2,https://tools.cdc.gov/api/v2/resources/media/4...,CDC Media
3,https://www.cdc.gov/mmwr/index2026.xml,CDC MMWR
4,https://emergency.cdc.gov/rss.asp,CDC Emergency Preparedness
...,...,...
84,https://www.theguardian.com/society/health/rss,Guardian Health
85,https://www.reutersagency.com/feed/?best-topic...,Reuters Healthcare
86,https://kffhealthnews.org/feed,KFF Health News
87,https://yourlocalepidemiologist.substack.com/feed,Your Local Epidemiologist


In [8]:
# ── Date-parsing helper ────────────────────────────────────────────────────
def parse_published(raw: str):
    """Try RFC 2822 (email.utils) then a set of common strptime formats."""
    if not raw:
        return None
    try:
        return email.utils.parsedate_to_datetime(raw).replace(tzinfo=None)
    except Exception:
        pass
    for fmt in (
        "%a, %d %b %Y %H:%M:%S %z",
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S%z",
        "%Y-%m-%d %H:%M:%S",
    ):
        try:
            dt = datetime.datetime.strptime(raw.strip(), fmt)
            return dt.replace(tzinfo=None)
        except ValueError:
            pass
    return None


# ── Keyword-relevance filter ───────────────────────────────────────────────
_kw_lower = [k.lower() for k in HEALTH_KEYWORDS]

def is_health_relevant(title: str, summary: str = "") -> bool:
    """Return True if at least one health keyword appears in title or summary."""
    haystack = (title + " " + summary).lower()
    return any(kw in haystack for kw in _kw_lower)


# ── Hardened feed fetcher ──────────────────────────────────────────────────
def fetch_feed(feed_info: dict) -> list:
    """
    Resilient RSS/Atom fetcher with:
      • retry/backoff via requests
      • SSL verification disabled (tolerates expired/mismatched certs)
      • malformed-XML recovery via lxml
      • HTML fallback + feed autodiscovery
    Returns a list of entry dicts (may be empty).
    """
    url   = feed_info["url"]
    label = feed_info.get("label", url)
    entries = []

    session = build_session()

    try:
        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT,
            verify=False,          # tolerate SSL cert issues
            allow_redirects=True,
        )
        content_type = response.headers.get("Content-Type", "")
        raw = response.content

        # ── Primary parse ──────────────────────────────────────────────────
        parsed = feedparser.parse(raw)

        # ── XML recovery pass ──────────────────────────────────────────────
        if not parsed.entries:
            repaired = recover_malformed_xml(raw)
            parsed = feedparser.parse(repaired)

        # ── HTML fallback + autodiscovery ──────────────────────────────────
        if not parsed.entries and "html" in content_type.lower():
            for discovered_url in discover_feed_from_html(raw, response.url):
                try:
                    nested = session.get(
                        discovered_url, timeout=REQUEST_TIMEOUT, verify=False
                    )
                    nested_feed = feedparser.parse(nested.content)
                    if nested_feed.entries:
                        parsed = nested_feed
                        break
                except Exception:
                    continue

        if not parsed.entries:
            return entries

        # ── Extract entries ────────────────────────────────────────────────
        for entry in parsed.entries[:MAX_ENTRIES_PER_FEED]:
            title   = entry.get("title", "").strip()
            link    = entry.get("link",  "").strip()
            summary = entry.get("summary", "")
            raw_pub = entry.get("published", "")

            if not title or not link:
                continue
            if not is_health_relevant(title, summary):
                continue

            entries.append({
                "source":    label,
                "title":     title,
                "link":      link,
                "summary":   summary[:280] if summary else "",
                "published": parse_published(raw_pub),
                "raw_date":  raw_pub,
            })

    except Exception as exc:
        print(f"  ⚠  {label}: {exc}")

    return entries


# ── Fetch all feeds ────────────────────────────────────────────────────────
print("Fetching feeds…")
all_entries = []
for feed_info in RSS_FEEDS:
    fetched = fetch_feed(feed_info)
    all_entries.extend(fetched)
    status = f"{len(fetched)} items" if fetched else "0 items / skipped"
    print(f"  {'✓' if fetched else '–'}  {feed_info['label']}: {status}")

# ── Deduplicate on canonical URL ───────────────────────────────────────────
_NEWS_COLS = ["source", "title", "link", "summary", "published", "raw_date"]

if all_entries:
    news_df = (
        pd.DataFrame(all_entries)
        .drop_duplicates(subset=["link"])
        .sort_values("published", ascending=False, na_position="last")
        .reset_index(drop=True)
    )
else:
    news_df = pd.DataFrame(columns=_NEWS_COLS)

print(f"\n✓ {len(news_df)} unique health-relevant articles across {len(RSS_FEEDS)} feeds")
if not news_df.empty:
    news_df[["source", "title", "published"]].head(10)
else:
    print("  (No articles — feeds may be blocked or all errored in this environment.)")


Fetching feeds…
  –  CDC Newsroom: 0 items / skipped
  –  NIH News Releases: 0 items / skipped
  –  CDC Media: 0 items / skipped
  –  CDC MMWR: 0 items / skipped
  –  CDC Emergency Preparedness: 0 items / skipped
  –  NIH News Releases: 0 items / skipped
  –  NIH Director Blog: 0 items / skipped
  ✓  FDA Recalls: 9 items
  –  FDA Press: 0 items / skipped
  –  CMS News: 0 items / skipped
  –  HHS News: 0 items / skipped
  –  ASPR Preparedness: 0 items / skipped
  –  SAMHSA: 0 items / skipped
  –  HRSA: 0 items / skipped
  –  AHRQ: 0 items / skipped
  –  NIAID: 0 items / skipped
  –  NIMH: 0 items / skipped
  –  Network for Public Health Law: 0 items / skipped
  ✓  Health Affairs: 15 items
  –  Commonwealth Fund: 0 items / skipped
  ✓  KFF Policy: 12 items
  –  Pew Health Care: 0 items / skipped
  –  NCSL: 0 items / skipped
  –  PubMed News: 0 items / skipped
  ✓  National Governors Association: 1 items
  –  ASTHO: 0 items / skipped
  –  NACCHO: 0 items / skipped
  –  Council of State Go

## 3 · Tracker DataFrames

Fill in rows manually as you document meetings, research, and policy actions.

In [9]:
# ── Meetings & agreements tracker ─────────────────────────────────────────
# Add rows as dicts to the list below, then re-run.
MEETING_ROWS: list[dict] = [
    {
        "Date":            "2026-01-14",
        "Alliance":        "GPHA",
        "Topic":           "Inaugural GPHA Summit — strategic priorities",
        "States Involved": "CA, CO, CT, DE, GU, HI, IL, MA, MD, NC, NJ, NY, OR, RI, WA",
        "Key Outcomes":    "Adopted shared disease-surveillance data standards; formed Vaccines and Emergency Preparedness work groups",
        "Source":          "https://www.govsforhealth.org/",
    },
    {
        "Date":            "2026-02-20",
        "Alliance":        "WCHA",
        "Topic":           "West Coast respiratory-disease coordination call",
        "States Involved": "CA, OR, WA",
        "Key Outcomes":    "Agreed on joint wastewater-surveillance protocol; aligned influenza vaccine messaging timeline for spring",
        "Source":          "",
    },
    {
        "Date":            "2026-03-05",
        "Alliance":        "GPHA",
        "Topic":           "Emergency Preparedness Work Group — stockpile policy",
        "States Involved": "IL, NY, CA, MD, MA",
        "Key Outcomes":    "Drafted shared strategic-stockpile access agreement; legal review pending in three states",
        "Source":          "",
    },
    {
        "Date":            "2026-04-10",
        "Alliance":        "NEPHC",
        "Topic":           "Outbreak response tabletop exercise",
        "States Involved": "Northeast regional states",
        "Key Outcomes":    "Identified gaps in cross-border laboratory data sharing; tasked sub-group with interoperability roadmap",
        "Source":          "",
    },
    {
        "Date":            "2026-05-01",
        "Alliance":        "ASTHO",
        "Topic":           "Annual State Health Officials Conference",
        "States Involved": "All U.S. states and territories",
        "Key Outcomes":    "Endorsed joint federal-advocacy priorities on CDC funding; released workforce-development framework",
        "Source":          "https://www.astho.org/",
    },
]

meetings_df = pd.DataFrame(
    MEETING_ROWS or [{}],
    columns=["Date", "Alliance", "Topic", "States Involved", "Key Outcomes", "Source"],
)
if not MEETING_ROWS:
    meetings_df = meetings_df.iloc[0:0]   # keep schema, no phantom row

print(f"Meetings tracker: {len(meetings_df)} entries")
meetings_df


Meetings tracker: 5 entries


,Date,Alliance,Topic,States Involved,Key Outcomes,Source
0,2026-01-14,GPHA,Inaugural GPHA Summit — strategic priorities,"CA, CO, CT, DE, GU, HI, IL, MA, MD, NC, NJ, NY...",Adopted shared disease-surveillance data stand...,https://www.govsforhealth.org/
1,2026-02-20,WCHA,West Coast respiratory-disease coordination call,"CA, OR, WA",Agreed on joint wastewater-surveillance protoc...,
2,2026-03-05,GPHA,Emergency Preparedness Work Group — stockpile ...,"IL, NY, CA, MD, MA",Drafted shared strategic-stockpile access agre...,
3,2026-04-10,NEPHC,Outbreak response tabletop exercise,Northeast regional states,Identified gaps in cross-border laboratory dat...,
4,2026-05-01,ASTHO,Annual State Health Officials Conference,All U.S. states and territories,Endorsed joint federal-advocacy priorities on ...,https://www.astho.org/


In [10]:
# ── Research & scientific findings tracker ─────────────────────────────────
RESEARCH_ROWS: list[dict] = [
    {
        "Date":             "2026-01-28",
        "Alliance":         "GPHA",
        "Research Topic":   "Cross-state wastewater surveillance sensitivity for novel respiratory pathogens",
        "Institutions":     "UCLA Fielding School, UW School of Public Health",
        "Policy Relevance": "Supports WCHA joint wastewater-surveillance protocol; informs CDC partnership proposal",
        "Link":             "",
    },
    {
        "Date":             "2026-02-14",
        "Alliance":         "WCHA",
        "Research Topic":   "mRNA booster efficacy variation across climate zones",
        "Institutions":     "UCSF, Oregon Health & Science University, Fred Hutchinson Cancer Center",
        "Policy Relevance": "Informs WCHA vaccine harmonization plan and cold-chain storage standards",
        "Link":             "",
    },
    {
        "Date":             "2026-03-22",
        "Alliance":         "NEPHC",
        "Research Topic":   "Laboratory interoperability standards for interstate outbreak reporting",
        "Institutions":     "Johns Hopkins Bloomberg School of Public Health, Yale School of Public Health",
        "Policy Relevance": "Foundation for NEPHC interoperability roadmap assigned at March tabletop exercise",
        "Link":             "",
    },
    {
        "Date":             "2026-04-05",
        "Alliance":         "ASTHO",
        "Research Topic":   "Public health workforce retention and pipeline development",
        "Institutions":     "George Washington Milken Institute of Public Health",
        "Policy Relevance": "Directly cited in ASTHO workforce-development framework released at May conference",
        "Link":             "https://www.astho.org/",
    },
]

research_df = pd.DataFrame(
    RESEARCH_ROWS or [{}],
    columns=["Date", "Alliance", "Research Topic", "Institutions", "Policy Relevance", "Link"],
)
if not RESEARCH_ROWS:
    research_df = research_df.iloc[0:0]

print(f"Research tracker: {len(research_df)} entries")
research_df


Research tracker: 4 entries


,Date,Alliance,Research Topic,Institutions,Policy Relevance,Link
0,2026-01-28,GPHA,Cross-state wastewater surveillance sensitivit...,"UCLA Fielding School, UW School of Public Health",Supports WCHA joint wastewater-surveillance pr...,
1,2026-02-14,WCHA,mRNA booster efficacy variation across climate...,"UCSF, Oregon Health & Science University, Fred...",Informs WCHA vaccine harmonization plan and co...,
2,2026-03-22,NEPHC,Laboratory interoperability standards for inte...,Johns Hopkins Bloomberg School of Public Healt...,Foundation for NEPHC interoperability roadmap ...,
3,2026-04-05,ASTHO,Public health workforce retention and pipeline...,George Washington Milken Institute of Public H...,Directly cited in ASTHO workforce-development ...,https://www.astho.org/


In [11]:
# ── Policy & regulatory tracker ────────────────────────────────────────────
POLICY_ROWS: list[dict] = [
    {
        "Date":                "2026-01-20",
        "Alliance":            "GPHA",
        "Policy Area":         "Vaccine mandate harmonization",
        "Action":              "Joint letter to HHS requesting federal alignment guidance",
        "States":              "All 15 GPHA member states/territories",
        "Federal Interaction": "HHS Secretary response pending; CDC convened technical working group",
        "Source":              "https://www.govsforhealth.org/",
    },
    {
        "Date":                "2026-02-08",
        "Alliance":            "WCHA",
        "Policy Area":         "Climate-health preparedness",
        "Action":              "Tri-state executive order aligning extreme-heat public-health response protocols",
        "States":              "CA, OR, WA",
        "Federal Interaction": "Shared with FEMA regional office; cited in HHS climate-health brief",
        "Source":              "",
    },
    {
        "Date":                "2026-03-15",
        "Alliance":            "GPHA",
        "Policy Area":         "Disease surveillance data sharing",
        "Action":              "Adopted common data-sharing agreement template for member states",
        "States":              "CA, IL, NY, MD, MA, WA — initial signatories",
        "Federal Interaction": "CDC offered to host shared data infrastructure; MOU in negotiation",
        "Source":              "https://www.govsforhealth.org/",
    },
    {
        "Date":                "2026-04-22",
        "Alliance":            "ASTHO",
        "Policy Area":         "Federal public health funding",
        "Action":              "Congressional testimony supporting restoration of CDC Prevention Fund",
        "States":              "Representatives from 12 state health agencies",
        "Federal Interaction": "Submitted formal comment to Senate HELP Committee; hearing scheduled",
        "Source":              "https://www.astho.org/",
    },
    {
        "Date":                "2026-05-10",
        "Alliance":            "NEPHC",
        "Policy Area":         "Cross-border emergency authority",
        "Action":              "Drafted model interstate compact language for emergency health declarations",
        "States":              "Northeast regional states",
        "Federal Interaction": "Reviewed by HHS Office of General Counsel; no federal objection noted",
        "Source":              "",
    },
]

policy_df = pd.DataFrame(
    POLICY_ROWS or [{}],
    columns=["Date", "Alliance", "Policy Area", "Action", "States", "Federal Interaction", "Source"],
)
if not POLICY_ROWS:
    policy_df = policy_df.iloc[0:0]

print(f"Policy tracker: {len(policy_df)} entries")
policy_df


Policy tracker: 5 entries


,Date,Alliance,Policy Area,Action,States,Federal Interaction,Source
0,2026-01-20,GPHA,Vaccine mandate harmonization,Joint letter to HHS requesting federal alignme...,All 15 GPHA member states/territories,HHS Secretary response pending; CDC convened t...,https://www.govsforhealth.org/
1,2026-02-08,WCHA,Climate-health preparedness,Tri-state executive order aligning extreme-hea...,"CA, OR, WA",Shared with FEMA regional office; cited in HHS...,
2,2026-03-15,GPHA,Disease surveillance data sharing,Adopted common data-sharing agreement template...,"CA, IL, NY, MD, MA, WA — initial signatories",CDC offered to host shared data infrastructure...,https://www.govsforhealth.org/
3,2026-04-22,ASTHO,Federal public health funding,Congressional testimony supporting restoration...,Representatives from 12 state health agencies,Submitted formal comment to Senate HELP Commit...,https://www.astho.org/
4,2026-05-10,NEPHC,Cross-border emergency authority,Drafted model interstate compact language for ...,Northeast regional states,Reviewed by HHS Office of General Counsel; no ...,


## 4 · Alliance–State Network Visualization

In [12]:
# ── Build graph ────────────────────────────────────────────────────────────
G = nx.Graph()

ALLIANCE_COLOR = "#1a4a2e"
STATE_COLOR    = "#c94c2e"
EDGE_COLOR     = "#b0a898"

for _, row in alliance_df.iterrows():
    abbr  = row["Abbreviation"]
    G.add_node(abbr, node_type="alliance", full=row["Alliance"])
    states = [s.strip() for s in row["Member States/Territories"].split(",") if s.strip()]
    for state in states:
        # Truncate long state descriptions for readability
        label = state if len(state) <= 25 else state[:22] + "…"
        if not G.has_node(label):
            G.add_node(label, node_type="state")
        G.add_edge(abbr, label)

alliance_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "alliance"]
state_nodes    = [n for n, d in G.nodes(data=True) if d.get("node_type") == "state"]

print(f"Graph: {len(alliance_nodes)} alliance nodes, {len(state_nodes)} state/territory nodes, {G.number_of_edges()} edges")

# ── Layout & draw ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 11))
ax.set_facecolor("#f4f1eb")
fig.patch.set_facecolor("#f4f1eb")

pos = nx.spring_layout(G, seed=42, k=2.8)

# Edges
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=EDGE_COLOR, alpha=0.55, width=1.2)

# Alliance nodes (larger, dark green)
nx.draw_networkx_nodes(
    G, pos, nodelist=alliance_nodes, ax=ax,
    node_color=ALLIANCE_COLOR, node_size=900, linewidths=1.5, edgecolors="#fff",
)
# State nodes (smaller, terracotta)
nx.draw_networkx_nodes(
    G, pos, nodelist=state_nodes, ax=ax,
    node_color=STATE_COLOR, node_size=320, linewidths=0.8, edgecolors="#fff", alpha=0.85,
)

# Labels — alliance names bold & larger
nx.draw_networkx_labels(
    G, pos, labels={n: n for n in alliance_nodes}, ax=ax,
    font_size=9, font_color="#fff", font_weight="bold",
)
nx.draw_networkx_labels(
    G, pos, labels={n: n for n in state_nodes}, ax=ax,
    font_size=6.5, font_color="#3a2f22",
)

# Legend
legend_handles = [
    mpatches.Patch(color=ALLIANCE_COLOR, label="Alliance / Compact"),
    mpatches.Patch(color=STATE_COLOR,    label="Member State / Territory"),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=8,
          framealpha=0.9, edgecolor=EDGE_COLOR)

ax.set_title("Alliance–State Membership Network", fontsize=13, pad=14,
             fontfamily="serif", color="#1c1b17")
ax.axis("off")
plt.tight_layout()

# Save to PNG for embedding
GRAPH_PNG = pathlib.Path("alliance_network.png")
plt.savefig(GRAPH_PNG, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"✓ Network graph saved → {GRAPH_PNG}")


Graph: 4 alliance nodes, 17 state/territory nodes, 20 edges
✓ Network graph saved → alliance_network.png


## 5 · Membership Count Chart

In [13]:
# ── Membership counts (rough — counts comma-separated tokens) ──────────────
def count_members(member_str: str) -> int:
    return len([s for s in member_str.split(",") if s.strip()])

alliance_df["Member Count"] = alliance_df["Member States/Territories"].apply(count_members)

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor("#f4f1eb")
ax.set_facecolor("#f4f1eb")

abbrs  = alliance_df["Abbreviation"]
counts = alliance_df["Member Count"]
bars   = ax.barh(abbrs, counts, color="#1a4a2e", height=0.5)

for bar, val in zip(bars, counts):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=9, color="#1c1b17")

ax.set_xlabel("Member States / Territories (approx.)", fontsize=9)
ax.set_title("Alliance Membership Counts", fontsize=12, fontfamily="serif", pad=10)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
ax.set_xlim(0, counts.max() + 4)
plt.tight_layout()

BAR_PNG = pathlib.Path("membership_counts.png")
plt.savefig(BAR_PNG, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"✓ Bar chart saved → {BAR_PNG}")


✓ Bar chart saved → membership_counts.png


## 6 · HTML Dashboard Export

In [14]:
# ── Encode images as base64 so the HTML is fully self-contained ────────────
import base64

def png_to_data_uri(path: pathlib.Path) -> str:
    if not path.exists():
        return ""
    with open(path, "rb") as f:
        return "data:image/png;base64," + base64.b64encode(f.read()).decode()

graph_uri = png_to_data_uri(GRAPH_PNG)
bar_uri   = png_to_data_uri(BAR_PNG)

print(f"Network graph  : {'embedded' if graph_uri else 'MISSING'} ({GRAPH_PNG})")
print(f"Membership bars: {'embedded' if bar_uri   else 'MISSING'} ({BAR_PNG})")


Network graph  : embedded (alliance_network.png)
Membership bars: embedded (membership_counts.png)


In [15]:
# ── DataFrame → JSON helpers ───────────────────────────────────────────────
def df_to_json(df: pd.DataFrame) -> str:
    """Serialize DataFrame to JSON, converting datetime → ISO string."""
    def _conv(obj):
        if isinstance(obj, (datetime.datetime, datetime.date)):
            return obj.isoformat()
        return str(obj)
    return json.dumps(df.to_dict(orient="records"), default=_conv)


# ── Source list for the dashboard Sources tab ──────────────────────────────
SOURCE_LIST = [
    {"label": "Governors for Health (GPHA)",       "url": "https://www.govsforhealth.org/"},
    {"label": "GovFacts – About GPHA",             "url": "https://govfacts.org/health-healthcare/public-health/about-the-governors-public-health-alliance/"},
    {"label": "ASTHO",                             "url": "https://www.astho.org/"},
    {"label": "CDC Media RSS",                     "url": "https://www.cdc.gov/media/rss.xml"},
    {"label": "NIH News Releases",                 "url": "https://www.nih.gov/news-events/news-releases/feed"},
    {"label": "NYT Health",                        "url": "https://rss.nytimes.com/services/xml/rss/nyt/Health.xml"},
    {"label": "KFF Health News",                   "url": "https://kffhealthnews.org/feed"},
    {"label": "ProPublica Main Feed",              "url": "https://feeds.propublica.org/propublica/main"},
    {"label": "Your Local Epidemiologist",         "url": "https://yourlocalepidemiologist.substack.com/feed"},
]

print("✓ Data ready for export")
print(f"  Alliances : {len(alliance_df)}")
print(f"  News items: {len(news_df)}")
print(f"  Meetings  : {len(meetings_df)}")
print(f"  Research  : {len(research_df)}")
print(f"  Policy    : {len(policy_df)}")


✓ Data ready for export
  Alliances : 4
  News items: 190
  Meetings  : 5
  Research  : 4
  Policy    : 5


In [16]:
# ── Build HTML ─────────────────────────────────────────────────────────────
# The template below is self-contained: CSS, JS, and all data are inline.
# Images are base64-embedded so the file works offline.

GENERATED_AT = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

ALLIANCE_JSON  = df_to_json(alliance_df)
NEWS_JSON      = df_to_json(news_df)
MEETINGS_JSON  = df_to_json(meetings_df)
RESEARCH_JSON  = df_to_json(research_df)
POLICY_JSON    = df_to_json(policy_df)
SOURCES_JSON   = json.dumps(SOURCE_LIST)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Public Health Compacts Monitor · 2026</title>
  <link rel="preconnect" href="https://fonts.googleapis.com" />
  <link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@400;500&family=Libre+Baskerville:ital,wght@0,400;0,700;1,400&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet" />
  <style>
    :root {{
      --bg:#f4f1eb;--surface:#fffef9;--border:#d6cebc;--accent:#1a4a2e;
      --accent2:#c94c2e;--text:#1c1b17;--muted:#7a7468;--tag-bg:#e8e3d5;
      --mono:'DM Mono',monospace;--serif:'Libre Baskerville',Georgia,serif;
      --sans:'DM Sans',sans-serif;
    }}
    *,*::before,*::after{{box-sizing:border-box;margin:0;padding:0}}
    body{{font-family:var(--sans);background:var(--bg);color:var(--text);min-height:100vh}}
    header{{background:var(--accent);color:#fff;padding:2rem 2.5rem 1.6rem;display:flex;align-items:flex-end;gap:2rem;flex-wrap:wrap}}
    header h1{{font-family:var(--serif);font-size:clamp(1.4rem,2.5vw,2.1rem);font-weight:700;letter-spacing:-0.01em;line-height:1.15}}
    header h1 span{{display:block;font-style:italic;font-weight:400;opacity:0.75;font-size:0.75em;margin-top:0.2em}}
    .header-meta{{margin-left:auto;font-family:var(--mono);font-size:0.72rem;opacity:0.65;text-align:right}}
    .header-meta strong{{display:block;opacity:1;font-size:0.8rem}}
    nav{{background:var(--surface);border-bottom:1px solid var(--border);padding:0 2.5rem;display:flex;gap:0;overflow-x:auto}}
    .tab{{padding:0.9rem 1.4rem;font-size:0.82rem;font-weight:500;color:var(--muted);cursor:pointer;border-bottom:3px solid transparent;white-space:nowrap;transition:color .15s,border-color .15s;background:none;border-top:none;border-left:none;border-right:none;font-family:var(--sans)}}
    .tab:hover{{color:var(--text)}}.tab.active{{color:var(--accent);border-bottom-color:var(--accent)}}
    main{{max-width:1300px;margin:0 auto;padding:2rem 2.5rem 4rem}}
    .panel{{display:none}}.panel.active{{display:block}}
    .section-head{{display:flex;align-items:baseline;gap:1rem;margin-bottom:1.2rem;padding-bottom:0.6rem;border-bottom:1px solid var(--border)}}
    .section-head h2{{font-family:var(--serif);font-size:1.25rem;font-weight:700}}
    .section-head .count{{font-family:var(--mono);font-size:0.72rem;color:var(--muted);background:var(--tag-bg);padding:0.15em 0.5em;border-radius:3px}}
    .card-grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(340px,1fr));gap:1.2rem;margin-bottom:2.5rem}}
    .card{{background:var(--surface);border:1px solid var(--border);border-radius:6px;padding:1.4rem;transition:box-shadow .15s}}
    .card:hover{{box-shadow:0 4px 16px rgba(0,0,0,.08)}}
    .card-name{{font-family:var(--serif);font-size:1rem;font-weight:700;color:var(--accent);margin-bottom:0.35rem}}
    .card-name a{{color:inherit;text-decoration:none}}.card-name a:hover{{text-decoration:underline}}
    .card-abbr{{font-family:var(--mono);font-size:0.7rem;color:var(--muted);margin-bottom:0.8rem}}
    .card dl{{display:grid;grid-template-columns:auto 1fr;gap:0.3rem 0.8rem;font-size:0.82rem}}
    .card dt{{color:var(--muted);font-weight:500;white-space:nowrap}}.card dd{{color:var(--text)}}
    .tags{{display:flex;flex-wrap:wrap;gap:0.3rem;margin-top:0.7rem}}
    .tag{{font-family:var(--mono);font-size:0.67rem;background:var(--tag-bg);color:var(--muted);padding:0.15em 0.55em;border-radius:3px}}
    .tag.collab{{background:#dde8e2;color:var(--accent)}}
    .table-wrap{{overflow-x:auto;margin-bottom:2rem}}
    table{{width:100%;border-collapse:collapse;font-size:0.82rem;background:var(--surface)}}
    thead th{{background:var(--accent);color:#fff;padding:0.65rem 0.9rem;text-align:left;font-family:var(--mono);font-size:0.7rem;letter-spacing:0.04em;white-space:nowrap;cursor:pointer;user-select:none}}
    thead th::after{{content:' ↕';opacity:0.4}}thead th.asc::after{{content:' ↑';opacity:1}}thead th.desc::after{{content:' ↓';opacity:1}}
    tbody tr{{border-bottom:1px solid var(--border)}}tbody tr:hover{{background:#f9f6ee}}
    tbody td{{padding:0.6rem 0.9rem;vertical-align:top}}tbody td:first-child{{font-weight:600}}
    tbody td a{{color:var(--accent);text-decoration:none;word-break:break-all}}tbody td a:hover{{text-decoration:underline;color:var(--accent2)}}
    .filter-bar{{display:flex;gap:0.6rem;flex-wrap:wrap;margin-bottom:1.2rem;align-items:center}}
    .filter-bar input,.filter-bar select{{padding:0.5rem 0.8rem;border:1px solid var(--border);border-radius:4px;font-family:var(--sans);font-size:0.82rem;background:var(--surface)}}
    .filter-bar input{{flex:1 1 220px}}.filter-bar input:focus{{outline:2px solid var(--accent);border-color:transparent}}
    .news-list{{display:flex;flex-direction:column;gap:0.6rem}}
    .news-item{{background:var(--surface);border:1px solid var(--border);border-radius:5px;padding:0.85rem 1.1rem;display:flex;flex-direction:column;gap:0.25rem;transition:box-shadow .12s}}
    .news-item:hover{{box-shadow:0 2px 10px rgba(0,0,0,.07)}}
    .news-item a{{font-family:var(--serif);font-size:0.92rem;color:var(--accent);text-decoration:none;font-weight:700;line-height:1.35}}
    .news-item a:hover{{text-decoration:underline;color:var(--accent2)}}
    .news-meta{{display:flex;gap:0.8rem;align-items:center;flex-wrap:wrap}}
    .news-source{{font-family:var(--mono);font-size:0.68rem;color:var(--muted);background:var(--tag-bg);padding:0.1em 0.45em;border-radius:3px}}
    .news-date{{font-family:var(--mono);font-size:0.68rem;color:var(--muted)}}
    .no-results{{font-size:0.85rem;color:var(--muted);padding:1.5rem 0;text-align:center;font-style:italic}}
    .img-wrap{{background:var(--surface);border:1px solid var(--border);border-radius:6px;padding:1.2rem;margin-bottom:1.5rem;text-align:center}}
    .img-wrap img{{max-width:100%;height:auto;border-radius:4px}}
    .img-wrap p{{font-family:var(--mono);font-size:0.7rem;color:var(--muted);margin-top:0.6rem}}
    .empty-state{{background:var(--surface);border:1px solid var(--border);border-radius:6px;padding:2rem;text-align:center;color:var(--muted);font-size:0.85rem}}
    .empty-state strong{{display:block;color:var(--text);margin-bottom:0.3rem}}
    footer{{text-align:center;font-family:var(--mono);font-size:0.68rem;color:var(--muted);padding:2rem;border-top:1px solid var(--border)}}
  </style>
</head>
<body>
<header>
  <div>
    <h1>Public Health Compacts Monitor<span>Interstate Alliance Tracking · United States</span></h1>
  </div>
  <div class="header-meta"><strong>{GENERATED_AT}</strong>Last generated</div>
</header>

<nav>
  <button class="tab active" data-panel="alliances">Alliances</button>
  <button class="tab" data-panel="news">News Feed</button>
  <button class="tab" data-panel="meetings">Meetings</button>
  <button class="tab" data-panel="research">Research</button>
  <button class="tab" data-panel="policy">Policy</button>
  <button class="tab" data-panel="sources">Sources</button>
</nav>

<main>

<!-- ALLIANCES -->
<div class="panel active" id="panel-alliances">
  <div class="section-head"><h2>Alliance Overview</h2><span class="count" id="alliance-count"></span></div>
  <div class="card-grid" id="alliance-cards"></div>
  <div class="section-head" style="margin-top:2rem"><h2>Comparison Table</h2></div>
  <div class="table-wrap">
    <table id="alliance-table">
      <thead><tr>
        <th>Alliance</th><th>Founded</th><th>Members</th>
        <th>Goals</th><th>Focus Areas</th><th>Collaborations</th><th>Governance</th>
      </tr></thead>
      <tbody id="alliance-tbody"></tbody>
    </table>
  </div>
  <div class="section-head" style="margin-top:2rem"><h2>Visualizations</h2></div>
  <div class="img-wrap">
    <img src="{graph_uri}" alt="Alliance–State Membership Network" />
    <p>Alliance–State Membership Network</p>
  </div>
  <div class="img-wrap">
    <img src="{bar_uri}" alt="Membership Count by Alliance" />
    <p>Membership Count by Alliance</p>
  </div>
</div>

<!-- NEWS FEED -->
<div class="panel" id="panel-news">
  <div class="section-head"><h2>Latest News</h2><span class="count" id="news-count"></span></div>
  <div class="filter-bar">
    <input type="text" id="news-search" placeholder="Search headlines…" />
    <select id="news-source-filter"><option value="">All sources</option></select>
  </div>
  <div class="news-list" id="news-list"></div>
  <div class="no-results" id="news-empty" style="display:none">No matching articles.</div>
</div>

<!-- MEETINGS -->
<div class="panel" id="panel-meetings">
  <div class="section-head"><h2>Meetings &amp; Agreements</h2><span class="count" id="meetings-count"></span></div>
  <div id="meetings-content"></div>
</div>

<!-- RESEARCH -->
<div class="panel" id="panel-research">
  <div class="section-head"><h2>Research &amp; Scientific Findings</h2><span class="count" id="research-count"></span></div>
  <div id="research-content"></div>
</div>

<!-- POLICY -->
<div class="panel" id="panel-policy">
  <div class="section-head"><h2>Policy &amp; Regulatory Tracking</h2><span class="count" id="policy-count"></span></div>
  <div id="policy-content"></div>
</div>

<!-- SOURCES -->
<div class="panel" id="panel-sources">
  <div class="section-head"><h2>Reference Sources</h2></div>
  <div class="card" style="max-width:640px">
    <p style="font-size:0.82rem;color:var(--muted);margin-bottom:1rem">RSS feeds and reference links used by the monitoring notebook.</p>
    <ul id="source-list" style="list-style:none;display:flex;flex-direction:column;gap:0.5rem;font-size:0.83rem"></ul>
  </div>
</div>

</main>
<footer>Public Health Compacts Monitor · generated {GENERATED_AT} · data via CDC, NIH, NYT Health, NPR, KFF Health News, ProPublica</footer>

<script>
const ALLIANCES = {ALLIANCE_JSON};
const NEWS      = {NEWS_JSON};
const MEETINGS  = {MEETINGS_JSON};
const RESEARCH  = {RESEARCH_JSON};
const POLICY    = {POLICY_JSON};
const SOURCES   = {SOURCES_JSON};

// ── Alliance cards & table ─────────────────────────────────────────────────
document.getElementById("alliance-count").textContent = ALLIANCES.length + " alliances";
const cardGrid = document.getElementById("alliance-cards");
const tbody    = document.getElementById("alliance-tbody");

ALLIANCES.forEach(a => {{
  const abbr     = a["Abbreviation"] || "";
  const parenIdx = a["Alliance"].indexOf("(");
  const fullName = (parenIdx > -1 ? a["Alliance"].slice(0, parenIdx) : a["Alliance"]).trim();
  const website  = a["Website"] || "";
  const nameHtml = website
    ? `<a href="${{website}}" target="_blank" rel="noopener">${{fullName}}</a>`
    : fullName;
  const focusTags  = (a["Work Groups / Focus Areas"] || "").split(",")
    .map(s => `<span class="tag">${{s.trim()}}</span>`).join("");
  const collabTags = (a["Cross-Collaborations"] || "").split(",")
    .map(s => `<span class="tag collab">${{s.trim()}}</span>`).join("");

  cardGrid.innerHTML += `
    <div class="card">
      <div class="card-name">${{nameHtml}}</div>
      <div class="card-abbr">${{abbr}} · Founded: ${{a["Founded"]}}</div>
      <dl>
        <dt>Members</dt><dd>${{a["Member States/Territories"]}}</dd>
        <dt>Goals</dt><dd>${{a["Primary Goals"]}}</dd>
        <dt>Governance</dt><dd>${{a["Governance Notes"] || "—"}}</dd>
      </dl>
      <div class="tags">${{focusTags}}</div>
      <div class="tags">${{collabTags}}</div>
    </div>`;

  const wsCell = website
    ? `<a href="${{website}}" target="_blank" rel="noopener">${{website}}</a>`
    : "—";
  tbody.innerHTML += `<tr>
    <td>${{a["Alliance"]}}</td>
    <td>${{a["Founded"]}}</td>
    <td>${{a["Member States/Territories"]}}</td>
    <td>${{a["Primary Goals"]}}</td>
    <td>${{a["Work Groups / Focus Areas"]}}</td>
    <td>${{a["Cross-Collaborations"]}}</td>
    <td>${{a["Governance Notes"] || "—"}}</td>
  </tr>`;
}});

// Sortable table
let sortState = {{col: null, dir: 1}};
document.querySelectorAll("#alliance-table thead th").forEach((th, i) => {{
  th.addEventListener("click", () => {{
    const rows = Array.from(tbody.querySelectorAll("tr"));
    document.querySelectorAll("#alliance-table thead th")
      .forEach(h => h.classList.remove("asc","desc"));
    sortState.dir = sortState.col === i ? -sortState.dir : 1;
    sortState.col = i;
    th.classList.add(sortState.dir === 1 ? "asc" : "desc");
    rows.sort((a,b) => a.cells[i].textContent.trim()
      .localeCompare(b.cells[i].textContent.trim()) * sortState.dir);
    rows.forEach(r => tbody.appendChild(r));
  }});
}});

// ── News feed ──────────────────────────────────────────────────────────────
const newsList   = document.getElementById("news-list");
const newsEmpty  = document.getElementById("news-empty");
const newsSearch = document.getElementById("news-search");
const newsFilter = document.getElementById("news-source-filter");
const newsCount  = document.getElementById("news-count");

const sources = [...new Set(NEWS.map(n => n.source))].filter(Boolean).sort();
sources.forEach(s => {{
  const o = document.createElement("option"); o.value = o.textContent = s;
  newsFilter.appendChild(o);
}});

function fmtDate(iso) {{
  if (!iso) return "";
  try {{ return new Date(iso).toLocaleDateString("en-US", {{month:"short",day:"numeric",year:"numeric"}}); }}
  catch(e) {{ return iso.slice(0,10); }}
}}

function renderNews() {{
  const q = newsSearch.value.toLowerCase();
  const src = newsFilter.value;
  const shown = NEWS.filter(n =>
    (!q   || n.title.toLowerCase().includes(q)) &&
    (!src || n.source === src)
  );
  newsCount.textContent = shown.length + " of " + NEWS.length + " items";
  if (shown.length === 0 && NEWS.length === 0) {{
    newsList.innerHTML = `<div class="empty-state"><strong>No news loaded.</strong>
      Run the notebook and re-export to populate the feed.</div>`;
    newsEmpty.style.display = "none";
    return;
  }}
  newsList.innerHTML = shown.map(n => `
    <div class="news-item">
      <div class="news-meta">
        <span class="news-source">${{n.source || "Unknown"}}</span>
        ${{n.published ? `<span class="news-date">${{fmtDate(n.published)}}</span>` : ""}}
      </div>
      <a href="${{n.link}}" target="_blank" rel="noopener noreferrer">${{n.title}}</a>
    </div>`).join("");
  newsEmpty.style.display = shown.length ? "none" : "block";
}}

renderNews();
newsSearch.addEventListener("input", renderNews);
newsFilter.addEventListener("change", renderNews);

// ── Tracker tables helper ──────────────────────────────────────────────────
function renderTracker(data, containerId, countId, emptyMsg) {{
  const countEl = document.getElementById(countId);
  const wrap    = document.getElementById(containerId);
  countEl.textContent = data.length + " entries";
  if (!data.length) {{
    wrap.innerHTML = `<div class="empty-state"><strong>No entries yet.</strong> ${{emptyMsg}}</div>`;
    return;
  }}
  const cols = Object.keys(data[0]);
  const rows = data.map(row => `<tr>${{cols.map(c => {{
    const val = row[c] || "—";
    const linked = String(val).startsWith("http")
      ? `<a href="${{val}}" target="_blank" rel="noopener">${{val}}</a>`
      : val;
    return `<td>${{linked}}</td>`;
  }}).join("")}}</tr>`).join("");
  wrap.innerHTML = `<div class="table-wrap"><table>
    <thead><tr>${{cols.map(c=>`<th>${{c}}</th>`).join("")}}</tr></thead>
    <tbody>${{rows}}</tbody>
  </table></div>`;
}}

renderTracker(MEETINGS, "meetings-content", "meetings-count",
  "Add rows to <code>MEETING_ROWS</code> in the notebook.");
renderTracker(RESEARCH, "research-content", "research-count",
  "Add rows to <code>RESEARCH_ROWS</code> in the notebook.");
renderTracker(POLICY,   "policy-content",   "policy-count",
  "Add rows to <code>POLICY_ROWS</code> in the notebook.");

// ── Sources panel ──────────────────────────────────────────────────────────
const srcList = document.getElementById("source-list");
SOURCES.forEach(s => {{
  srcList.innerHTML += `<li>
    <a href="${{s.url}}" target="_blank" rel="noopener"
       style="color:var(--accent);font-family:var(--mono);font-size:0.78rem">${{s.label || s.url}}</a>
    <span style="font-family:var(--mono);font-size:0.68rem;color:var(--muted);
                 display:block;margin-top:0.1rem;word-break:break-all">${{s.url}}</span>
  </li>`;
}});

// ── Tab switching ──────────────────────────────────────────────────────────
document.querySelectorAll(".tab").forEach(tab => {{
  tab.addEventListener("click", () => {{
    document.querySelectorAll(".tab").forEach(t => t.classList.remove("active"));
    document.querySelectorAll(".panel").forEach(p => p.classList.remove("active"));
    tab.classList.add("active");
    document.getElementById("panel-" + tab.dataset.panel).classList.add("active");
  }});
}});
</script>
</body>
</html>"""

OUTPUT_HTML.write_text(html, encoding="utf-8")
print(f"\n✓ Dashboard written → {OUTPUT_HTML.resolve()}")
print(f"  File size: {OUTPUT_HTML.stat().st_size / 1024:.1f} KB")



✓ Dashboard written → C:\Users\Dylanator\Documents\truth_audit\Interstate Compacts\Public Health\public_health_dashboard.html
  File size: 520.7 KB


## 8 · Weekly Brief Export

Runs automatically after the HTML export. Writes `weekly_brief_input.txt` — paste its contents into the NarroVue Weekly Brief Generator (Step 1).


In [17]:
# ── Weekly brief export ────────────────────────────────────────────────────
# Writes a plain-text summary file you can paste directly into the
# NarroVue Weekly Brief Generator (Step 1 · Paste Notebook Output).

import textwrap

BRIEF_TXT = pathlib.Path("weekly_brief_input.txt")

lines = []
lines.append(f"NarroVue · Public Health Compacts Monitor")
lines.append(f"Run date : {GENERATED_AT}")
lines.append(f"Alliances: {len(alliance_df)}")
lines.append(f"News items fetched: {len(news_df)}")
lines.append("")

# ── Meetings ──────────────────────────────────────────────────────────────
lines.append(f"MEETINGS ({len(meetings_df)} entries)")
lines.append("-" * 48)
if meetings_df.empty:
    lines.append("  (none)")
else:
    for _, r in meetings_df.iterrows():
        lines.append(f"  {r.get('Date','')} | {r.get('Alliance','')} | {r.get('Topic','')}")
        if r.get("Key Outcomes"):
            for ln in textwrap.wrap(r["Key Outcomes"], 72):
                lines.append(f"    → {ln}")
        if r.get("States Involved"):
            lines.append(f"    States: {r['States Involved']}")
        if r.get("Source"):
            lines.append(f"    Source: {r['Source']}")
        lines.append("")

# ── Research ──────────────────────────────────────────────────────────────
lines.append(f"RESEARCH ({len(research_df)} entries)")
lines.append("-" * 48)
if research_df.empty:
    lines.append("  (none)")
else:
    for _, r in research_df.iterrows():
        lines.append(f"  {r.get('Date','')} | {r.get('Alliance','')} | {r.get('Research Topic','')}")
        if r.get("Institutions"):
            lines.append(f"    Institutions: {r['Institutions']}")
        if r.get("Policy Relevance"):
            for ln in textwrap.wrap(r["Policy Relevance"], 72):
                lines.append(f"    → {ln}")
        if r.get("Link"):
            lines.append(f"    Link: {r['Link']}")
        lines.append("")

# ── Policy ────────────────────────────────────────────────────────────────
lines.append(f"POLICY ({len(policy_df)} entries)")
lines.append("-" * 48)
if policy_df.empty:
    lines.append("  (none)")
else:
    for _, r in policy_df.iterrows():
        lines.append(f"  {r.get('Date','')} | {r.get('Alliance','')} | {r.get('Policy Area','')}")
        if r.get("Action"):
            for ln in textwrap.wrap(r["Action"], 72):
                lines.append(f"    Action: {ln}")
        if r.get("States"):
            lines.append(f"    States: {r['States']}")
        if r.get("Federal Interaction"):
            for ln in textwrap.wrap(r["Federal Interaction"], 72):
                lines.append(f"    Federal: {ln}")
        if r.get("Source"):
            lines.append(f"    Source: {r['Source']}")
        lines.append("")

# ── Top news headlines ─────────────────────────────────────────────────────
lines.append(f"TOP NEWS HEADLINES (up to 10 most recent)")
lines.append("-" * 48)
if news_df.empty:
    lines.append("  (no news fetched this run)")
else:
    for _, r in news_df.head(10).iterrows():
        date_str = r["published"].strftime("%Y-%m-%d") if r.get("published") and hasattr(r["published"], "strftime") else str(r.get("raw_date",""))[:10]
        lines.append(f"  {date_str} | {r.get('source','')} | {r.get('title','')}")
        lines.append(f"    {r.get('link','')}")
        lines.append("")

output_text = "\n".join(lines)
BRIEF_TXT.write_text(output_text, encoding="utf-8")
print(f"\n✓ Weekly brief input written → {BRIEF_TXT.resolve()}")
print(f"  File size: {BRIEF_TXT.stat().st_size} bytes")
print()
print(output_text[:800] + ("\n  …" if len(output_text) > 800 else ""))



✓ Weekly brief input written → C:\Users\Dylanator\Documents\truth_audit\Interstate Compacts\Public Health\weekly_brief_input.txt
  File size: 6435 bytes

NarroVue · Public Health Compacts Monitor
Run date : 2026-05-27 07:29
Alliances: 4
News items fetched: 190

MEETINGS (5 entries)
------------------------------------------------
  2026-01-14 | GPHA | Inaugural GPHA Summit — strategic priorities
    → Adopted shared disease-surveillance data standards; formed Vaccines and
    → Emergency Preparedness work groups
    States: CA, CO, CT, DE, GU, HI, IL, MA, MD, NC, NJ, NY, OR, RI, WA
    Source: https://www.govsforhealth.org/

  2026-02-20 | WCHA | West Coast respiratory-disease coordination call
    → Agreed on joint wastewater-surveillance protocol; aligned influenza
    → vaccine messaging timeline for spring
    States: CA, OR, WA

  2026-03-05 | GPHA | Emergency Preparedness Work Group — stockpile policy
    → Drafted shared strategic-st
  …


## 7 · Manual Search Targets

Queries to run periodically in search engines or news aggregators to supplement RSS monitoring.


In [18]:
SEARCH_QUERIES = [
    "Governors Public Health Alliance meeting",
    "West Coast Health Alliance vaccine policy",
    "NEPHC public health collaboration",
    "multi-state vaccine coordination 2026",
    "interstate outbreak preparedness compact",
    "state public health alliance emergency response",
    "public health interstate agreement",
    "ASTHO state health official statement",
    "GPHA press release",
]

print("Suggested search queries:\n")
for i, q in enumerate(SEARCH_QUERIES, 1):
    print(f"  {i:2d}. {q}")


Suggested search queries:

   1. Governors Public Health Alliance meeting
   2. West Coast Health Alliance vaccine policy
   3. NEPHC public health collaboration
   4. multi-state vaccine coordination 2026
   5. interstate outbreak preparedness compact
   6. state public health alliance emergency response
   7. public health interstate agreement
   8. ASTHO state health official statement
   9. GPHA press release
